# SGLang + AuditKIT: serving a Cohere model and evaluating it

SGLang serves a model behind an OpenAI-compatible HTTP API, and AuditKIT's `api:` backend evaluates it. The two run in **separate environments**: SGLang and vLLM pin their GPU stacks exactly, and the pins disagree (see `docs/SGLANG.md`).
- AuditKIT's environment only needs `auditkit[requests]`.
- SGLang gets its own venv inside this runtime.

| § | What |
|---|---|
| 1 | Build the SGLang venv on **Python 3.12** (via `uv`; SGLang's `outlines_core` pin has no 3.13 wheels). The SGLang version is picked from the NVIDIA driver: ≥ 580 → `sglang>=0.5.18` (CUDA 13), otherwise `0.5.10` (the last CUDA 12 build). |
| 2 | Launch the server with **Tiny Aya** (`cohere2`) and wait until it's healthy. |
| 3 | Check both environments with `check_compat()`. |
| 4 | Generation eval through `api:`: multilingual QA, with latency and throughput. |
| 5 | Tool-call eval through `api:` in **prompt mode**, since Tiny Aya's template has no tool slot. |
| 6 | Shut down. |

**Runtime:** a Colab **L4 or A100** (a T4 works with `DTYPE = "half"`). If flashinfer can't JIT-build its kernels, the launch cell retries once with the Triton attention backend. SGLang is Linux-only.

**Access:** Tiny Aya is gated, so add the `HF_TOKEN` secret, plus `GITHUB_TOKEN`. To use an open model instead, set `MODEL = "Qwen/Qwen2.5-1.5B-Instruct"`.

In [1]:
# ---- settings ----
import os
MODEL = os.environ.get("NB_MODEL", "CohereLabs/tiny-aya-global")
PORT = 30000
DTYPE = "bfloat16"              # "half" on a T4
EXTRA_SERVER_ARGS = []          # the launch cell falls back to ["--attention-backend", "triton", ...] if flashinfer's kernel build fails
SERVER_TIMEOUT_S = 1200

In [2]:
# ---- install (Colab) / reuse (local checkout) ----
# Colab: clones AuditKIT at REPO_BRANCH and installs it with the extras this notebook needs.
# Local: uses the auditkit already importable (e.g. the repo's .venv kernel).
# AuditKIT itself only needs [requests] here; SGLang is installed into its own venv below.
import os, sys, subprocess
REPO_BRANCH = "feat/rag-agent-evals"   # #1's branch: #9, #10 and every #15 / #27 fix are merged here
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import userdata
    for key in ("GITHUB_TOKEN", "HF_TOKEN"):
        try:
            os.environ[key] = userdata.get(key)
        except Exception:
            pass
    repo_url = f"https://{os.environ['GITHUB_TOKEN']}@github.com/aryapratinavseth/AuditKIT-Internal.git"
    if not os.path.isdir("/content/AuditKIT-Internal"):
        subprocess.run(["git", "clone", "-q", "--branch", REPO_BRANCH, repo_url, "/content/AuditKIT-Internal"], check=True)
    else:   # a reused runtime: move the existing clone to the branch head, never run stale code
        subprocess.run(["git", "-C", "/content/AuditKIT-Internal", "fetch", "-q", repo_url, REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", "/content/AuditKIT-Internal", "checkout", "-q", "-B", REPO_BRANCH, "FETCH_HEAD"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-e",
                    "/content/AuditKIT-Internal[requests]"], check=True)
    sys.path.insert(0, "/content/AuditKIT-Internal/src")

if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
else:
    print("HF_TOKEN not set: gated Cohere models (Tiny Aya, Aya Expanse, Aya Vision) will fail to download.")

# The Hugging Face token from any source (the HF_TOKEN secret, a huggingface_hub.login() or a cached token),
# exported so the gated-model check and the SGLang server process both see it.
try:
    from huggingface_hub import get_token
    if not os.environ.get("HF_TOKEN") and get_token():
        os.environ["HF_TOKEN"] = get_token()
except Exception:
    pass

In [3]:
import json, signal, subprocess, sys, time
import requests
import auditkit as ak
from auditkit import RunConfig, Sample
SGL_ENV = "/content/sglang-env" if os.path.isdir("/content") else os.path.expanduser("~/sglang-env")
SGL_PY = f"{SGL_ENV}/bin/python"
BASE_URL = f"http://127.0.0.1:{PORT}"
API_BASE = f"{BASE_URL}/v1"
print("auditkit", ak.__version__, "| SGLang venv:", SGL_ENV)

auditkit 1.2.0 | SGLang venv: /content/sglang-env


## 1 · The SGLang environment

In [4]:
def driver_major():
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"],
                             capture_output=True, text=True, check=True).stdout
        return int(out.split(".")[0])
    except (OSError, subprocess.CalledProcessError, ValueError):
        return None

drv = driver_major()
SGLANG_SPEC = "sglang>=0.5.18,<0.6" if drv is None or drv >= 580 else "sglang==0.5.10"
print("NVIDIA driver:", drv, "->", SGLANG_SPEC)
# SGLang's venv runs Python 3.12: sglang pins outlines==0.1.11 -> outlines_core 0.1.26, which has Linux wheels
# only up to cp312. On Colab's Python 3.13, pip would try to compile it (Rust) and fail. uv fetches a standalone
# CPython 3.12 and --seed puts pip in the venv. An existing venv on another Python (e.g. a failed 3.13 attempt)
# is rebuilt; a good one is reused.
SGL_PYTHON = "3.12"

def env_python():
    if not os.path.exists(SGL_PY):
        return None
    r = subprocess.run([SGL_PY, "-c", "import sys; print('%d.%d' % sys.version_info[:2])"], capture_output=True, text=True)
    return r.stdout.strip() or None

if env_python() != SGL_PYTHON:
    print("building the SGLang venv on Python", SGL_PYTHON, "(found:", env_python(), ")")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run([sys.executable, "-m", "uv", "venv", "--python", SGL_PYTHON, "--seed", "--clear", SGL_ENV], check=True)
print("SGLang venv python:", env_python())

def pip(*args):
    r = subprocess.run([SGL_PY, "-m", "pip", "install", "-q", *args], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"pip install {' '.join(args)} failed; pip said:", (r.stderr or r.stdout)[-3000:])

pip("--upgrade", "pip")
pip(SGLANG_SPEC)           # first install downloads a few GB (torch, flashinfer, kernels)
pip("ninja")               # flashinfer JIT-compiles some attention kernels on first use and calls `ninja`
ver = subprocess.run([SGL_PY, "-c", "import sglang, transformers, torch; print('sglang', sglang.__version__, "
                      "'| transformers', transformers.__version__, '| torch', torch.__version__)"], capture_output=True, text=True)
print(ver.stdout or ver.stderr[-2000:])

NVIDIA driver: 580 -> sglang>=0.5.18,<0.6
building the SGLang venv on Python 3.12 (found: None )
SGLang venv python: 3.12
sglang 0.5.20 | transformers 5.12.1 | torch 2.13.0+cu130



## 2 · Launch the server

In [5]:
LOG_PATH = f"{SGL_ENV}/server.log"
# The server runs from the SGLang venv, so its bin/ (ninja, and the venv's own scripts) goes first on PATH:
# flashinfer JIT-builds attention kernels on first use and runs `ninja` by name.
server_env = {**os.environ, "PATH": f"{SGL_ENV}/bin:" + os.environ.get("PATH", "")}   # HF_TOKEN reaches the server
if "CUDA_HOME" not in server_env and os.path.isdir("/usr/local/cuda"):
    server_env["CUDA_HOME"] = "/usr/local/cuda"
# If flashinfer's JIT build still fails (e.g. Colab's nvcc doesn't match torch's CUDA 13), relaunch once with
# the Triton attention backend and PyTorch sampling, which need no kernel compilation.
NO_JIT_ARGS = ["--attention-backend", "triton", "--sampling-backend", "pytorch"]
JIT_ERRORS = ("ninja", "nvcc", "flashinfer/jit", "build_and_load", "CUDA_HOME")

def log_tail(n=40):
    with open(LOG_PATH) as f:
        return "".join(f.readlines()[-n:])

def launch(extra):
    cmd = [SGL_PY, "-m", "sglang.launch_server", "--model-path", MODEL,
           "--host", "127.0.0.1", "--port", str(PORT), "--dtype", DTYPE, *extra]
    print("launching:", " ".join(cmd[2:]))
    return subprocess.Popen(cmd, stdout=open(LOG_PATH, "w"), stderr=subprocess.STDOUT,
                            start_new_session=True, env=server_env)

def wait_healthy(proc):
    """True once /health answers 200; False if the server exits; raises on timeout."""
    deadline = time.time() + SERVER_TIMEOUT_S
    while True:
        if proc.poll() is not None:
            return False
        try:
            if requests.get(f"{BASE_URL}/health", timeout=5).status_code == 200:
                return True
        except requests.RequestException:
            pass
        if time.time() > deadline:
            raise TimeoutError("server not healthy in time:\n" + log_tail())
        time.sleep(10)

server = launch(EXTRA_SERVER_ARGS)
if not wait_healthy(server):
    tail = log_tail(200)
    if "--attention-backend" in EXTRA_SERVER_ARGS or not any(k in tail for k in JIT_ERRORS):
        raise RuntimeError("SGLang server exited:\n" + log_tail())
    print("flashinfer kernel build failed; retrying with", NO_JIT_ARGS)
    EXTRA_SERVER_ARGS = [*EXTRA_SERVER_ARGS, *NO_JIT_ARGS]
    server = launch(EXTRA_SERVER_ARGS)
    if not wait_healthy(server):
        raise RuntimeError("SGLang server exited (Triton backend too):\n" + log_tail())
print("server healthy:", requests.get(f"{API_BASE}/models", timeout=10).json())

launching: sglang.launch_server --model-path CohereLabs/tiny-aya-global --host 127.0.0.1 --port 30000 --dtype bfloat16
server healthy: {'object': 'list', 'data': [{'id': 'CohereLabs/tiny-aya-global', 'object': 'model', 'created': 1790619706, 'owned_by': 'sglang', 'root': 'CohereLabs/tiny-aya-global', 'parent': None, 'max_model_len': 8192}]}


## 3 · Both environments pass `check_compat()`

In [6]:
from auditkit.compat import check_compat
import auditkit.compat as compat_module
report = check_compat()
print("AuditKIT env:\n", report)
sgl = subprocess.run([SGL_PY, compat_module.__file__, "--cuda"], capture_output=True, text=True)
print("SGLang env:\n", sgl.stdout or sgl.stderr)
print("AuditKIT env ok:", report.ok, "| SGLang env ok:", sgl.returncode == 0)

AuditKIT env:
 AuditKit compatibility report

  python             3.13.15
  platform           Linux x86_64
  glibc              2.39
  auditkit           1.2.0
  torch              2.11.0+cu128
  transformers       5.16.1
  vllm               -
  sglang             -
  flashinfer-python  -
  xgrammar           -
  llguidance         -
  openai             2.54.0
  litellm            -

  [INFO ] sglang: neither sglang nor vllm is installed here. Serving SGLang over HTTP and evaluating it with the api: backend needs only auditkit[requests] in this env.

OK
SGLang env:
 AuditKit compatibility report

  python             3.12.3
  platform           Linux x86_64
  glibc              2.39
  auditkit           -
  torch              2.13.0
  transformers       5.12.1
  vllm               -
  sglang             0.5.20
  flashinfer-python  0.6.18
  xgrammar           0.2.1
  llguidance         1.8.0
  openai             2.6.1
  litellm            -

  [WARN ] transformers: transformers 5.12

## 4 · Generation eval through `api:`

**Always pass `api_key`.** Without it, `APIModel` would fall back to `OPENAI_API_KEY` and send your real key to this server. SGLang ignores the key, so `"EMPTY"` is fine.

In [7]:
QA = [("What is the capital of France? Answer with one word.", "Paris"),
      ("Translate to Spanish, reply with the translation only: thank you", "gracias"),
      ("Wie viele Tage hat eine Woche? Nur die Zahl.", "7"),
      ("日本の首都はどこですか？一語で答えてください。", "東京"),
      ("What is 12 times 12? Answer with the number only.", "144"),
      ("¿Cuál es la capital de Perú? Responde con una palabra.", "Lima")]
qa = [Sample(input=q, target=a) for q, a in QA]
gen = ak.evaluate(qa, model=f"api:{MODEL}", api_base=API_BASE, api_key="EMPTY",
                  scorers=["quasi_exact_match", "f1_score", "chrf"],
                  config=RunConfig(temperature=0.0, max_tokens=48, concurrency=4))
print("headline:", {k: round(v, 3) for k, v in gen.headline.items()}, "| errors:", gen.errors)
for p in gen.predictions:
    print(f"  {p.expected!r:>12} <- {(p.raw_output or '').strip()[:60]!r}")
print(gen.summary())       # includes latency / throughput measured over HTTP

headline: {'quasi_exact_match': 0.833, 'f1_score': 0.333, 'chrf': 0.701} | errors: []
       'Paris' <- 'Paris.'
     'gracias' <- 'Gracias'
           '7' <- '7'
          '東京' <- '東京'
         '144' <- '12'
        'Lima' <- 'Lima.'
run efe804b25fe4116d (fingerprint efe804b25fe4116d)
  quasi_exact_match: 0.8333 ± 0.1521  (n=6)
  f1_score: 0.3333 ± 0.1925  (n=6)
  chrf: 0.7012 ± 0.1185  (n=6)
  perf: latency=9419ms (n=1 calls) | throughput=0.64 req/s | 2.2 out-tok/s, 253.9 total-tok/s
  model: CohereLabs/tiny-aya-global (not a local model -- no measurable size)
  tokens: 2,370 in + 21 out = 2,391 total (pass pricing to .cost() for a $ figure)


## 5 · Tool calling through `api:` (prompt mode)

Tiny Aya's chat template has no tool slot, so SGLang has nothing to parse natively. `ToolCallAdapter(mode="prompt")` writes the schemas into the system message, and AuditKIT parses the `<tool_call>` blocks the model writes back. For a model with a native tool parser (e.g. Qwen2.5 with `--tool-call-parser qwen25`), use `ToolCallAdapter()` instead and the calls come back structured.

In [8]:
TOOLS = [{"type": "function", "function": {"name": "get_weather", "description": "Current weather for a city",
          "parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}}},
         {"type": "function", "function": {"name": "get_time", "description": "Current local time in a city",
          "parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}}}]
W = lambda c: {"name": "get_weather", "arguments": {"city": c}}
tool_tasks = [Sample(id="single", input="What's the weather in Paris?", tools=TOOLS, expected_tool_calls=[W("Paris")]),
              Sample(id="parallel", input="What's the weather in Paris and in Rome? Call the tool for both cities.",
                     tools=TOOLS, expected_tool_calls=[[W("Paris"), W("Rome")]]),
              Sample(id="no_tool", input="What is 2+2? Answer directly, no tools.", tools=TOOLS, expected_tool_calls=[]),
              Sample(id="pick", input="What time is it in Tokyo?", tools=TOOLS,
                     expected_tool_calls=[{"name": "get_time", "arguments": {"city": "Tokyo"}}])]
tools_run = ak.evaluate(tool_tasks, model=f"api:{MODEL}", api_base=API_BASE, api_key="EMPTY",
                        adapter=ak.ToolCallAdapter(mode="prompt"),
                        scorers=[ak.ToolCallF1(), ak.ParallelToolCalls(), ak.ToolCallValidity()],
                        config=RunConfig(temperature=0.0, max_tokens=256))
print("headline:", {k: round(v, 3) for k, v in tools_run.headline.items()}, "| errors:", tools_run.errors)
for p in tools_run.predictions:
    print(f"  {p.sample_id:9s} {(p.raw_output or '').strip()[:110]!r}")

headline: {'tool_call_precision': 0.75, 'tool_call_recall': 0.75, 'tool_call_f1': 0.75, 'tool_call_exact': 0.75, 'parallel_detection': 1.0, 'tool_call_validity': 1.0, 'parallel_recall': 1.0, 'parallel_precision': 1.0} | errors: []
  single    '<tool_call>{"name": "get_weather", "arguments": {"city": "Paris"}}</tool_call>'
  parallel  '<tool_call>{"name": "get_weather", "arguments": {"city": "Paris"}}</tool_call>\n<tool_call>{"name": "get_weathe'
  no_tool   '4'
  pick      '<tool_call>{"name": "get_weather", "arguments": {"city": "Tokyo"}}</tool_call>'


## 6 · Shut down

In [9]:
os.killpg(os.getpgid(server.pid), signal.SIGTERM)
server.wait(timeout=60)
print("server stopped")

server stopped
